In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "/content/drive/MyDrive/부산_BigData_공모전/01_raw_data/생활수요/일별 행정동 연령 생활인구 월별 일평균.xlsx"

df = pd.read_excel(file_path)

print("데이터 크기:", df.shape)
display(df.head())

print("\n컬럼")
print(df.columns.tolist())

In [ ]:
# 생활인구 원본의 결측·중복·범위와 데이터 구조를 한 번에 검증

print("===== 기본 정보 =====")
print("행 수:", len(df))
print("완전 중복:", df.duplicated().sum())

print("\n===== 결측치 =====")
print(df.isnull().sum())

print("\n===== 기간 =====")
print("최소:", df["기준년월"].min())
print("최대:", df["기준년월"].max())
print("기준년월 개수:", df["기준년월"].nunique())
print(sorted(df["기준년월"].unique()))

print("\n===== 행정동 =====")
print("행정동코드 개수:", df["행정동코드"].nunique())
print("행정동명 개수:", df["행정동명"].nunique())

print("\n===== 연령대 =====")
print(df["나이대"].value_counts())

print("\n===== 인구 변수 기초통계 =====")
pop_cols = ["평균주거인구수", "평균직장인구수", "평균방문인구수"]
print(df[pop_cols].describe())

print("\n===== 음수 확인 =====")
for col in pop_cols:
    print(f"{col}: {(df[col] < 0).sum()}건")

print("\n===== 행정동 × 월 × 연령대 중복 =====")
key_cols = ["기준년월", "행정동코드", "나이대"]
print("중복:", df.duplicated(subset=key_cols).sum())

print("\n===== 월별 행 수 =====")
monthly_count = df.groupby("기준년월").size()
print(monthly_count)

print("\n월별 행 수 종류:", monthly_count.unique())

In [ ]:
# 2025년 3월부터 추가된 행정동과 공백 행정동명을 확인

target_code = 2644059000

print("===== 추가 행정동 =====")
print(
    df[df["행정동코드"] == target_code]
    [["기준년월", "행정동코드", "행정동명"]]
    .drop_duplicates()
)

print("\n===== 행정동명 공백 =====")
blank_name = df["행정동명"].astype(str).str.strip().eq("")
print("공백 행 수:", blank_name.sum())

print("\n공백 행의 행정동코드")
print(df.loc[blank_name, "행정동코드"].value_counts())

In [ ]:
# 공백으로 들어온 행정동명을 공식 행정동 코드 기준으로 보정

df_clean = df.copy()

mask = (
    (df_clean["행정동코드"] == 2644059000)
    & df_clean["행정동명"].astype(str).str.strip().eq("")
)

df_clean.loc[mask, "행정동명"] = "신호동"

print("보정 건수:", mask.sum())
print("행정동명 공백:", df_clean["행정동명"].astype(str).str.strip().eq("").sum())
print("행정동명 개수:", df_clean["행정동명"].nunique())

In [ ]:
# 2025년 3월 이후 모든 행정동에 매월 6개 연령대가 존재하는지 확인

df_2025 = df_clean[df_clean["기준년월"] >= 202503].copy()

age_count = (
    df_2025.groupby(["기준년월", "행정동코드"])
    ["나이대"]
    .nunique()
)

print("분석 기간:", df_2025["기준년월"].min(), "~", df_2025["기준년월"].max())
print("행 수:", len(df_2025))
print("행정동 수:", df_2025["행정동코드"].nunique())

print("\n연령대 개수가 6개가 아닌 경우:")
print(age_count[age_count != 6])

print("\n월별 행정동 수:")
print(
    df_2025.groupby("기준년월")["행정동코드"].nunique()
)

## 생활인구 데이터 품질검증 결과

- 원본 데이터: 44,340건
- 기간: 2023.01 ~ 2025.12 (36개월)
- 결측치: 없음
- 완전 중복: 없음
- 인구 변수 음수값: 없음
- 연령대: 6개
  - 20대미만 / 20대 / 30대 / 40대 / 50대 / 60대이상

### 행정동 체계 확인

2023.01~2025.02에는 월별 205개 행정동이 존재하며,
2025.03부터 월별 206개 행정동이 존재한다.

강서구 신호동은 2025년 1월 1일 녹산동에서 분동되었으며,
생활인구 데이터에서는 2025.03부터 별도 행정동코드(`2644059000`)로 나타난다.

해당 코드의 행정동명 60건이 공백으로 제공되어 `신호동`으로 보정하였다.

2025.03~2025.12 기간에는
206개 행정동 × 6개 연령대 × 10개월 = 12,360건이 모두 존재함을 확인하였다.

### 분석기간 결정

**본 분석기간: 2025.03 ~ 2025.12**

206개 행정동이 동일한 행정구역 체계로 존재하는 공통기간이며,
최신 지역 생활수요를 비교하기에 적합하다고 판단하였다.

2023.01~2025.12 전체 자료는 필요한 경우 시계열 변화 확인을 위한 보조자료로 활용한다.

In [ ]:
# 2025.03~12 데이터를 행정동·연령대별 평균으로 집계
# 월별 변동을 완화하고 각 지역의 대표적인 생활인구 수준을 만든다.

pop_age = (
    df_2025
    .groupby(["행정동코드", "행정동명", "나이대"], as_index=False)
    [["평균주거인구수", "평균직장인구수", "평균방문인구수"]]
    .mean()
)

print("행 수:", len(pop_age))
print("행정동 수:", pop_age["행정동코드"].nunique())
print("연령대 수:", pop_age["나이대"].nunique())

display(pop_age.head(12))

In [ ]:
# 연령대를 합산하여 행정동별 주거·직장·방문 생활인구 규모를 생성

pop_dong = (
    pop_age
    .groupby(["행정동코드", "행정동명"], as_index=False)
    [["평균주거인구수", "평균직장인구수", "평균방문인구수"]]
    .sum()
)

pop_dong["전체활동인구"] = (
    pop_dong["평균주거인구수"]
    + pop_dong["평균직장인구수"]
    + pop_dong["평균방문인구수"]
)

display(
    pop_dong
    .sort_values("전체활동인구", ascending=False)
    .head(20)
)

In [ ]:
# 전처리가 완료된 생활인구 분석용 데이터를 저장

base_path = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/"

pop_age.to_csv(
    base_path + "생활인구_행정동연령별_202503_202512.csv",
    index=False,
    encoding="utf-8-sig"
)

pop_dong.to_csv(
    base_path + "생활인구_행정동별_202503_202512.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("연령별:", pop_age.shape)
print("행정동별:", pop_dong.shape)

# 생활인구 데이터 전처리 결과

## 1. 원본 데이터

- 데이터: 일별 행정동 연령 생활인구 월별 일평균
- 원본 데이터: 44,340건
- 기간: 2023.01 ~ 2025.12 (36개월)
- 변수: 기준년월, 행정동코드, 행정동명, 나이대, 평균주거인구수, 평균직장인구수, 평균방문인구수
- 연령대: 20대미만 / 20대 / 30대 / 40대 / 50대 / 60대이상

## 2. 데이터 품질검증

- 결측치 없음
- 완전 중복 없음
- 인구 변수 음수값 없음
- 행정동 × 기준년월 × 연령대 중복 없음

단, 행정동명에서 일반적인 NaN 결측은 없었으나
공백 문자열로 제공된 값 60건을 추가로 확인하였다.

## 3. 행정동 체계 확인 및 보정

2023.01~2025.02에는 월별 205개 행정동,
2025.03부터는 월별 206개 행정동이 존재한다.

강서구 신호동은 2025년 1월 1일 녹산동에서 분동되었으며,
생활인구 데이터에서는 2025.03부터 별도 행정동코드 `2644059000`으로 나타난다.

해당 코드의 행정동명 60건이 공백으로 제공되어
`신호동`으로 보정하였다.

보정 후 행정동명은 총 206개이다.

## 4. 본 분석기간 결정

**2025.03 ~ 2025.12**

해당 기간에는 매월 동일하게

- 206개 행정동
- 6개 연령대
- 월별 1,236건

이 모두 존재한다.

따라서 행정동 간 생활수요 비교의 기준을 통일하기 위해
2025.03~2025.12를 본 분석기간으로 사용한다.

2023.01~2025.12 전체 데이터는 필요한 경우
장기 변화 및 추세 확인을 위한 보조자료로 활용한다.

## 5. 분석용 데이터 생성

월별 변동을 완화하기 위해 2025.03~2025.12의 평균값을 이용하여
두 종류의 분석용 데이터를 생성하였다.

### 행정동 × 연령대 데이터

- 206개 행정동 × 6개 연령대 = 1,236건
- 주거인구 / 직장인구 / 방문인구의 기간 평균

### 행정동 데이터

- 206건
- 연령대를 합산한 행정동별 주거인구 / 직장인구 / 방문인구
- `전체활동인구` = 주거인구 + 직장인구 + 방문인구

`전체활동인구`는 실제 고유 인구수가 아니라
지역의 상대적인 생활활동 규모를 파악하기 위한 지표로 해석한다.

## 6. 다음 단계

생활수요 EDA에서 행정동별

- 주거 / 직장 / 방문 생활인구 특성
- 연령대별 생활수요
- 지역별 생활인구 유형

을 분석하고, 이를 바탕으로 공공서비스 공급 분석에 사용할
생활수요 특성을 결정한다.

In [ ]:
import os
import json
import glob

base = '/content/drive/MyDrive/부산_BigData_공모전'

# population preprocessing notebook 찾기
files = glob.glob(
    os.path.join(base, '**', '*population*preprocessing*.ipynb'),
    recursive=True
)

print('찾은 노트북:')
for f in files:
    print(f)

if not files:
    print('\n노트북을 찾지 못했습니다.')
else:
    path = files[0]

    with open(path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    keywords = [
        'groupby', 'mean(', '.mean', 'sum(',
        '평균주거인구수', '평균직장인구수', '평균방문인구수',
        '기준년월', 'month', '월'
    ]

    print('\n========== 관련 코드 셀 ==========')

    for i, cell in enumerate(nb['cells']):
        if cell.get('cell_type') != 'code':
            continue

        code = ''.join(cell.get('source', []))

        if any(k in code for k in keywords):
            print(f'\n--- CELL {i} ---')
            print(code)

In [ ]:
import os
import json
import glob

base = '/content/drive/MyDrive/부산_BigData_공모전/07_code'

notebooks = glob.glob(
    os.path.join(base, '**', '*.ipynb'),
    recursive=True
)

keywords = [
    'rank(', 'pct=True', 'percentile',
    'spearman', "method='spearman'", 'method="spearman"',
    '.corr(', '상관',
    '수요균형도', '활동특화도'
]

for path in notebooks:
    with open(path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    matched = []

    for i, cell in enumerate(nb.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue

        code = ''.join(cell.get('source', []))

        if any(k.lower() in code.lower() for k in keywords):
            matched.append((i, code))

    if matched:
        print('\n' + '=' * 100)
        print('NOTEBOOK:', path)
        print('=' * 100)

        for i, code in matched:
            print(f'\n--- CELL {i} ---')
            print(code)